In [129]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

from tqdm import tqdm
import emoji

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

In [131]:
df = pd.read_csv('data/faq_df.csv')
'''
df_clean = pd.DataFrame({
    'question_clean': pd.Series([None] * len(df), dtype="object"),
    'answer_clean': pd.Series([None] * len(df), dtype="object")
})
'''
df_clean = pd.read_csv('faq_df_clean.csv')
#df.head(7)
df_clean.head(10)

,question_clean,answer_clean
0,"Ребят, кто-нибудь может рассказать примерную ц...",Недавно друг всёлился за эту цену на космонавтах
1,"Ребят, а кто-то может сориентировать по цене о...",Нашли?
2,"Кто-нибудь знает, в каком формате здесь нужно ...","Да, я такой чайник."
3,"Кто-нибудь знает, в каком формате здесь нужно ...","Используется формат так называемой ""объективки..."
4,Спасибо за ответ. Какие в среднем расценки? Ск...,"Пожалуйста. Ну, зависит от навыков. Я точно не..."
5,По поводу самозанятого и ИП. Про проживание бо...,Вроде только для самозанятного
6,"Добрый вечер, может кто-нибудь посоветовать ме...",Добрый вечер. Можете поделиться резюме в личны...
7,"Добрый вечер, может кто-нибудь посоветовать ме...",NaN
8,"Всем привет. Коллеги, может, кто-то подскажёт:...",С макаронами это беда
9,Нужен временный vpn из для проверки работы. Ка...,Рекомендую воспользоваться Tailscale.


In [133]:
class LLM_normalizer:
    
    def __init__(self, model, API_KEY, SYSTEM_PROMPT, url):
        self.parser = StrOutputParser()
        self.model = model
        self.API_KEY = API_KEY
        self.SYSTEM_PROMPT = SYSTEM_PROMPT
        self.url = url
        self.cache_question = {}
        self.cache_answer = {}
        
    def normalize_text(self, llm, text, cache) -> str:
        '''try 3 times if fails => original text + check if question text is the same => use cache: no LLM call'''
        if text in cache:
            return cache[text]

        for attempt in range(3):
            message = [SystemMessage(content=self.SYSTEM_PROMPT), HumanMessage(content=str(text))]
            llm_response = llm.invoke(message)
            raw = self.parser.invoke(llm_response).strip()
            if raw:
                cache[text] = raw
                return raw

        cache[text] = text
        return text

    def normalize_dataframe_batches(self, df_source, df_target, model_name, batch_size=20) -> pd.DataFrame:
        '''we transform initial question, answer into normalized form => new dataFrame'''
        llm = ChatOpenAI(api_key=self.API_KEY, base_url=self.url, model=model_name)
        n = len(df_source)
        try:
            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                batch_idx = df_source.index[start:end]
            
                for idx in tqdm(batch_idx, desc=f'{model_name} batch {start} - {end}'):
                    if pd.isna(df_target.at[idx, 'question_clean']): 
                        question = df_source.at[idx, 'question']
                        question = emoji.replace_emoji(str(question), '')
                        df_target.at[idx, 'question_clean'] = self.normalize_text(llm, question, self.cache_question)
                        
                    if pd.isna(df_target.at[idx, 'answer_clean']): 
                        answer = df_source.at[idx, 'answer']
                        answer = emoji.replace_emoji(str(answer), '')
                        df_target.at[idx, 'answer_clean'] = self.normalize_text(llm, answer, self.cache_answer)
                    
        except KeyboardInterrupt:
            print('manual interrupt, return processed dataframe only')
            return df_target
            
        return df_target

In [135]:
SYSTEM_POMPT = '''Ты — модель для нормализации текста. Твоя задача — переписать входной текст в аккуратной, 
нейтральной, дружелюбной форме на русском языке, сохранив исходный смысл.

Правила:
1. Убирай эмодзи, смайлики, повторяющиеся знаки, растянутые слова, капслок.
2. Убирай грубую лексику, токсичность, агрессию. Заменяй на нейтральные формулировки.
3. Не используй официальный стиль. Пиши просто, естественно, как в нормальном дружелюбном диалоге.
4. Не добавляй новую информацию и не меняй факты.
5. Сохраняй вопрос, если это вопрос. Сохраняй структуру смысла.
6. Исправляй опечатки, пунктуацию и грамматику.
7. Не сокращай смысл и не делай текст слишком длинным.
8. Если текст пустой или состоит из мусора — верни пустую строку.

Выводи только очищенный текст, без пояснений.'''

In [137]:
model = 'qwen3-14b'
load_dotenv()
API_KEY = os.getenv("API_KEY")
url = 'https://bothub.chat/api/v2/openai/v1'

normalizer = LLM_normalizer(model, API_KEY, SYSTEM_POMPT, url)

In [139]:
faq_df_clean = normalizer.normalize_dataframe_batches(df_source=df, df_target=df_clean, model_name=model, batch_size=20)

qwen3-14b batch 4020 - 4034: 100%|██████████████| 14/14 [03:26<00:00, 14.74s/it]


In [140]:
df_clean.to_csv('faq_df_clean.csv', index=False, quoting=1)